In [ ]:
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting

corpus = tod.corpus.Corpus(
    treebank_path="/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_Naija-NSC",
    grew_pattern="pattern { X[upos=VERB|ADV|AUX]; Y[upos=VERB]; X <Y }",
    patterns_text_file="/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/other/patterns_naija.txt",
    min_occurrences=50,
    matrix_type="coverage"
)

Number of matches after filtering: 8845


In [8]:
hclustering = tod.clustering.HierarchicalClustering(corpus)

In [9]:
len(hclustering.clusters)

2

In [14]:
tsne = tod.dimension_reduction_classic.Tsne_corpus(corpus)
fig = tod.plotting.lexunit_scatter_plot(corpus, tsne)
fig.show()

In [15]:
pca = tod.dimension_reduction_classic.Pca_corpus(corpus)
fig = tod.plotting.lexunit_scatter_plot(corpus, pca)
fig.show()

In [18]:
pca.explained_variance

array([0.38596178, 0.08481792])

In [17]:
fig = tod.plotting.cluster_scatter_plot(corpus, pca, hclustering)
fig.show()

In [21]:
treebank_path="/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_Naija-NSC"
grew_pattern="pattern { X[upos=VERB|ADV|AUX]; Y[upos=VERB]; X <Y }"
patterns_text_file="/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/other/patterns_naija.txt"
min_occurrences=3

types_of_matrix = ["precision", "coverage", "PMI", "tf-idf", "geometric_mean"]
dim_reds = {}
corpora = {}
for type_of_matrix in types_of_matrix:
    corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    min_occurrences=min_occurrences,
    matrix_type=type_of_matrix
)
    corpora[type_of_matrix] = corpus
    dim_reds[type_of_matrix] = {"pca": tod.dimension_reduction_classic.Pca_corpus(corpus), "tsne": tod.dimension_reduction_classic.Tsne_corpus(corpus)}

Number of matches after filtering: 8845
Number of matches after filtering: 8845
Number of matches after filtering: 8845
Number of matches after filtering: 8845
Number of matches after filtering: 8845


In [22]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

def lexunit_scatter_plot_plotly(corpus, dim_reds):
    # Create a subplot layout with 2 rows (PCA, t-SNE) and 5 columns (matrix types)
    fig = make_subplots(
        rows=2, cols=5,
        subplot_titles=[f"{matrix_type} - PCA" for matrix_type in dim_reds.keys()] +
                       [f"{matrix_type} - t-SNE" for matrix_type in dim_reds.keys()],
        horizontal_spacing=0.05, vertical_spacing=0.15
    )

    # Iterate over the matrix types and dimension reduction methods
    for col, type_of_matrix in enumerate(dim_reds.keys(), start=1):
        for row, dim_red in enumerate(['pca', 'tsne'], start=1):
            # Get the dimension reduction object
            dimension_reduction = dim_reds[type_of_matrix][dim_red]

            # Prepare the data
            data = pd.DataFrame(
                {
                    "Component 1": dimension_reduction.reduced_matrix[:, 0],
                    "Component 2": dimension_reduction.reduced_matrix[:, 1],
                    "Lexical Unit": [
                        corpus.idx2lexunit(i)
                        for i in range(len(dimension_reduction.reduced_matrix))
                    ],
                }
            )

            # Add scatter plot to the subplot
            fig.add_trace(
                go.Scatter(
                    x=data["Component 1"],
                    y=data["Component 2"],
                    mode="markers",
                    marker=dict(size=8),
                    text=data["Lexical Unit"],  # Hover text
                    hovertemplate="Lexical Unit: %{text}<br>Component 1: %{x:.2f}<br>Component 2: %{y:.2f}<extra></extra>",
                ),
                row=row, col=col
            )

    # Update layout
    fig.update_layout(
        height=800,  # Adjust height
        width=1200,  # Adjust width
        title_text="Lexical Units Scatter Plots",
        showlegend=False,  # Hide legend
    )

    # Show the plot
    fig.show()

In [23]:
lexunit_scatter_plot_plotly(corpus, dim_reds)

In [24]:
for type_of_matrix, _ in dim_reds.items():
    print(f"{type_of_matrix}: {dim_reds[type_of_matrix]['pca'].explained_variance}")

precision: [0.30883833 0.16910675]
coverage: [0.38596178 0.08481792]
PMI: [0.0635798  0.06134543]
tf-idf: [0.40490851 0.09223183]
geometric_mean: [0.0326561  0.02463069]


In [26]:
corpora = {}
for type_of_matrix in types_of_matrix:
    corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    min_occurrences=min_occurrences,
    matrix_type=type_of_matrix
)
    corpora[type_of_matrix] = corpus

Number of matches after filtering: 8845
Number of matches after filtering: 8845
Number of matches after filtering: 8845
Number of matches after filtering: 8845
Number of matches after filtering: 8845
